In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [3]:
# SETUP

TRAINING_DATA = "/training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id',
    'best_tasks_free',
    'acad_tasks_rating',
    'best_tasks_select',
    'subopt_freq_rating',
    'subopt_tasks_select',
    'subopt_tasks_free',
    'evidence_freq_rating',
    'verify_freq_rating',
    'verify_method_free',
    'target'
    ]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [4]:
# EXPLORE DATA

df.isnull().sum()

,0
id,0
best_tasks_free,2
acad_tasks_rating,0
best_tasks_select,15
subopt_freq_rating,1
subopt_tasks_select,22
subopt_tasks_free,11
evidence_freq_rating,62
verify_freq_rating,4
verify_method_free,10


In [15]:
# """These are EQUIVALENT:

# Option 1: make_pipeline (shorter)
# text_pipeline = make_pipeline(
#     SimpleImputer(strategy="constant", fill_value=""),
#     TfidfVectorizer(max_features=2000)
# )

# Option 2: Pipeline (explicit)
# text_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy="constant", fill_value="")),
#     ('tfidf', TfidfVectorizer(max_features=2000))
# ])"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List



text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    # TFIDF manually (classical)
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # hardcoded stop_words param for now
        # TODO: try with max instead
        ('tfidf', TfidfVectorizer(stop_words='english'))
    ])

def preprocess_ord():

    return make_pipeline(
        OrdinalEncoder(categories= ord_categories, handle_unknown='use_encoded_value', unknown_value=np.nan),
        SimpleImputer(strategy="most_frequent")

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                         SimpleImputer(strategy="most_frequent"))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)

# Full pipeline
# added logistic regression as placeholder classifier for now
preprocessor = create_preprocessor()
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# ============ GRID SEARCH ON FULL PIPELINE ============

# param_grid = {
#     # Text TF-IDF parameters
#     'preprocessor__text__tfidf__max_features': [500, 1000, 2000, 3000],
#     'preprocessor__text__tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
#     'preprocessor__text__tfidf__min_df': [1, 2, 3, 4, 5],
#     'preprocessor__text__tfidf__max_df': [0.8, 0.9, 1.0],

#     # Classifier parameters
#     # TODO: change to SVM or RandomForest and tune their hyperparams
# }

# # Grid search
# grid_search = GridSearchCV(
#     full_pipeline,
#     param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=2
# )

# Fit on raw data (pipeline handles preprocessing)
X = df[text_cols + ord_cols + cat_cols]
y = df['target']

# df specified here
# grid_search.fit(X, y)

# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

In [8]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")




Best parameters: {'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3)}
Best accuracy: 0.605


In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)

print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:

def split_data(df):
    students = df['id'].unique()
    train_df, test_df = train_test_split(students, test_size=0.3, random_state=42)
